# ASQ Structural Modeling

Load ASQ train data and initialize SambaNova client for model usage.


In [1]:
from pathlib import Path
import json


base = Path.cwd()
train_path = base / '..' / 'data' / 'ASQ' / 'asq_train.json'
if not train_path.exists():
    train_path = base / 'data' / 'ASQ' / 'asq_train.json'

with open(train_path, 'r', encoding='utf-8') as f:
    asq_train = json.load(f)

print('ASQ train path:', train_path)
print('ASQ train length:', len(asq_train))


ASQ train path: /Users/shayan/Projects/NarrativeSimilarity/src/../data/ASQ/asq_train.json
ASQ train length: 8865


In [2]:
from sambanova import SambaNova

key_path = Path.cwd() / 'sambanova_key.txt'
if not key_path.exists():
    key_path = Path.cwd().parent / 'sambanova_key.txt'

if not key_path.exists():
    raise FileNotFoundError(f'SambaNova key file not found: {key_path}')

api_key = key_path.read_text(encoding='utf-8').strip()
if not api_key:
    raise ValueError(f'SambaNova key file is empty: {key_path}')

client = SambaNova(
    base_url='https://api.sambanova.ai/v1',
    api_key=api_key
)

print(f'SambaNova client initialized. Key file: {key_path}')


SambaNova client initialized. Key file: /Users/shayan/Projects/NarrativeSimilarity/sambanova_key.txt


In [3]:
models = client.models.list()
print(models)

ModelsResponse(data=[ModelResponse(id='DeepSeek-R1-0528', context_length=131072, max_completion_tokens=7168, object='model', owned_by='no-reply@sambanova.ai', pricing=Pricing(completion=7e-06, duration_per_hour=None, prompt=5e-06), sn_metadata={}), ModelResponse(id='DeepSeek-R1-Distill-Llama-70B', context_length=131072, max_completion_tokens=4096, object='model', owned_by='no-reply@sambanova.ai', pricing=Pricing(completion=1.4e-06, duration_per_hour=None, prompt=7e-07), sn_metadata={}), ModelResponse(id='DeepSeek-V3-0324', context_length=131072, max_completion_tokens=7168, object='model', owned_by='no-reply@sambanova.ai', pricing=Pricing(completion=4.5e-06, duration_per_hour=None, prompt=3e-06), sn_metadata={}), ModelResponse(id='DeepSeek-V3.1', context_length=131072, max_completion_tokens=7168, object='model', owned_by='no-reply@sambanova.ai', pricing=Pricing(completion=4.5e-06, duration_per_hour=None, prompt=3e-06), sn_metadata={}), ModelResponse(id='DeepSeek-V3.1-Terminus', context_

## Event Extraction (with `gpt-oss-120b`)

Extract main events from ASQ train narratives, then save results to a JSON file in `data/`.


In [5]:
import re

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


system_prompt = 'You are a helpful assistant and are annotating narratives for a research study.'

user_prompt_template = '''Definition of an event:

An event is a singular, concrete occurrence asserted to have happened at a particular time (and often place) in the story. Events are typically expressed by verbs, but may also be expressed by nouns or adjectives when they denote an occurrence. General habits, ongoing states, hypothetical situations, and background facts are NOT events unless they function as a specific story-like occurrence.

Task:

Extract the main events from the story below.

Guidelines:
- Extract only singular occurrences asserted to have happened.
- Do NOT extract general, habitual, or repeated actions.
- Do NOT extract background states or descriptions unless they clearly function as events.
- Negative events (things that failed or did not happen) may be included if they occur at a specific time and are narratively meaningful.
- Maintain chronological order.

Return ONLY a JSON array of strings. Do not include explanations or extra text.

Story:
{story}
'''

def parse_json_array(text):
    text = text.strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, list):
            return [str(x) for x in obj]
    except Exception:
        pass

    match = re.search(r'\[(.|\n|\r)*\]', text)
    if match:
        candidate = match.group(0)
        obj = json.loads(candidate)
        if isinstance(obj, list):
            return [str(x) for x in obj]

    raise ValueError('Model response did not contain a valid JSON array.')

def count_sentences(text):
    parts = re.split(r'[.!?]+', str(text).strip())
    parts = [p for p in parts if p.strip()]
    return len(parts)

# Set max_samples=None to run on the full dataset.
max_samples = None  # e.g., set to 100 for a quick test
records_all = asq_train if max_samples is None else asq_train[:max_samples]

# Keep only stories with >= 4 sentences.
records = [r for r in records_all if count_sentences(r.get('narrative', '')) >= 4]
print(f'Eligible records (>=4 sentences): {len(records)} / {len(records_all)}')

out_path = Path.cwd() / '..' / 'data' / 'asq_event_extraction_full.json'
if not out_path.parent.exists():
    out_path = Path.cwd() / 'data' / 'asq_event_extraction_full.json'
out_path.parent.mkdir(parents=True, exist_ok=True)

# Resume support: if output exists, continue from remaining IDs.
results = []
processed_ids = set()
if out_path.exists():
    try:
        with open(out_path, 'r', encoding='utf-8') as f:
            existing = json.load(f)
        if isinstance(existing, list):
            results = existing
            processed_ids = {str(x.get('id')) for x in existing if isinstance(x, dict) and x.get('id') is not None}
            print(f'Resuming from existing file with {len(results)} records at: {out_path}')
    except Exception as e:
        print(f'Could not read existing output, starting fresh. Reason: {e}')

remaining = [r for r in records if str(r.get('id')) not in processed_ids]
print(f'Remaining to process: {len(remaining)}')

save_every = 1000
processed_this_run = 0

iterator = tqdm(remaining, desc='Extracting events') if tqdm is not None else remaining
for rec in iterator:
    story = rec.get('narrative', '')
    user_prompt = user_prompt_template.format(story=story)

    resp = client.chat.completions.create(
        model='gpt-oss-120b',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ],
        temperature=0.0
    )

    raw = resp.choices[0].message.content
    events = parse_json_array(raw)

    results.append({
        'id': rec.get('id'),
        'label': rec.get('label'),
        'qn1': rec.get('qn1'),
        'qn2': rec.get('qn2'),
        'narrative': story,
        'events': events
    })

    processed_this_run += 1

    if processed_this_run % save_every == 0:
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        print(f'Checkpoint saved at {processed_this_run} new records (total saved: {len(results)}) -> {out_path}')

# Final save
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print('Saved:', out_path)
print('Total records saved:', len(results))
print('Sample event counts (first 10):', [len(r['events']) for r in results[:10]])

results[:2]


Eligible records (>=4 sentences): 7927 / 8865
Resuming from existing file with 5000 records at: /Users/shayan/Projects/NarrativeSimilarity/src/../data/asq_event_extraction_full.json
Remaining to process: 2927


Extracting events:   0%|          | 0/2927 [00:00<?, ?it/s]

Checkpoint saved at 1000 new records (total saved: 6000) -> /Users/shayan/Projects/NarrativeSimilarity/src/../data/asq_event_extraction_full.json
Checkpoint saved at 2000 new records (total saved: 7000) -> /Users/shayan/Projects/NarrativeSimilarity/src/../data/asq_event_extraction_full.json
Saved: /Users/shayan/Projects/NarrativeSimilarity/src/../data/asq_event_extraction_full.json
Total records saved: 7927
Sample event counts (first 10): [3, 2, 4, 3, 5, 3, 7, 3, 4, 2]


[{'id': '9tfn8g',
  'label': 0,
  'qn1': 'Is it rude to sneak out and ask a nurse if they have any advice?',
  'qn2': 'How to deal with loud roommate?',
  'narrative': "This might sound dumb but I've been in hospital 4 nights already and am meant to be going home tomorrow (I hope!). My roommate from the statt is lovely and I'm not sure if it was because I was so out of it or because she didn't snore but I haven't heard her snore before. Tonight she snores like a chainsaw and I CANNOT sleep at all. She's also developed a hacking cough that I think is related. It's my last night but if I don't sleep and am not well enough they might decide to keep me longer which I don't want. I also really need my sleep because I need to her better. The nurses are kinda bitchty and rude and I feel like they might say something loudly to or in front of my roommate if I complain. I can put up with a sleepless night normally but not when I'm this sick.",
  'events': ['The roommate snores like a chainsaw to